# 02 Кросс-таблица_Профиль_ЦА_досуг #

- Задача: Проанализировать досуг кастомной ЦА женщин 25-50 лет
- База: Профиль 2024
- Целевая группа: Ж 25-50
- Отчет: кросс-таблица
- Статистики: Universe, Col%
- Количество дробных знаков:	1	
- Тотал по столбцам		
- Переменные в строках: 			
- Заказ кофе, еды и продуктов за 6 месяцев
- Посещение за 6 месяцев
- Регулярное посещение
- Кинотеатры
- Театры, музеи, выставки
- Бары, клубы, дискотеки
- Поп-, рок-концерты, фестивали, open air
- Торгово-развлекательные центры
- Игровые центры и площадки (квесты, пейнтбол и др.)
- Салоны красоты, спа, сауны
- Курсы, мастер-классы (кроме учебы)  
- Парки, площади, городские пространства
- Тренажерные залы, бассейны, фитнес-центры
- Спортивные соревнования, матчи, игры
- Кафе, рестораны
- Кофейни
- Гастропространства, фудкорты
- Фаст-фуд, закусочные быстрого обслуживания


In [ ]:
import pandas as pd
from brandpulse_api.calc import BrandpulseCalc

# имя файла с данными
DB_FILE = 'brandpulse3.duckdb'

# создаем объект для расчетов
bp = BrandpulseCalc(db_file=DB_FILE)

Справочно исследуем свойства

In [ ]:
# поиск по описанию
# bp.find_property(text='Пол')

# поиск по имени
# bp.find_property(sys_name='SEX')

# поиск по комбинации имени и описания
# bp.find_property(text='Возраст', sys_name='SEX')

Справочно исследуем значения свойств

In [ ]:
# поиск по описанию
#bp.find_property_options(text='Покупали кофе')

# поиск по имени
#bp.find_property_options(sys_name='ORDERED_FOOD_TO')

# поиск по комбинации имени и описания
#bp.find_property_options(text='кофе', sys_name='COFFEE')

# поиск по имени свойства (символ ^ добавлен для поиска имен, начинающихся с этой строки)
#bp.find_property_options(property_sys_name='^FOOD_DELIVERY')

Задаем параметры расчета

In [ ]:
profile_id = [3900499002973208128] # id профиля 2024

# задаем параметры целевой группы (используя sys_name из базы)
demo = [
    {"property_name": "SEX", 
     "expression": "='FEMALE'"},
    {"property_name": "AGE_COUNT2", 
     "expression": " BETWEEN 'A25' AND 'A50' "}
]

Получаем данные расчета universe

In [ ]:
# пример расчета по атрибутам и их значениям
properties = [
    ("FOOD_DELIVERY", ['ORDERED_FOOD_TO', 'ORDERED_THE_DEL', 'BOUGHT_COFFEE_T', 'ONLSERV', 'NOTHING']),
    ("LEISURE_PLACES", ['MOVIE_THEATERS', 'THEATRES_CLASS', 'BARS_NIGHTCLUB', 'FESTIVALS_OPEN', 'SHOPPING_CENTER', 'PLAY_CENTERS_AN',
                        'BEAUTY_SALONS_', 'COURSES_MASTER', 'PARKS_SQUARES', 'SPORTS_CENTERS', 'SPORTS_COMPETIT', 'CAFE_RESTAURAN', 'COFFEE_SHOPS',
                        'GASTROPROSTRANS', 'FAST_FOOD_FAST', 'NOTHING']),
]

df1 = bp.calc_project_property_universe(project_ids=profile_id, demo=demo, properties=properties)
df1

In [ ]:
# пример расчета по атрибутам и их значениям в разбивке по категориям + добавляем постобработку датасета

properties = [
    ("USUAL_VIS",  
     ['YES'],
     ['KINOTEATRY', 'TEATRY_KLASSICH', 'BARY_NOCHNYE_KL', 'CONCERTS24', 'IGROVYE_TSENTRY_', 'SALONY_KRASOTY', 'KURSY_MASTER_K', 
      'PARKIPLOHADI', 'SPORTIVNYE_TSENT', 'SPORTIVNYE_SORE', 'KAFE_RESTORANY', 'KOFEJNI', 'GASTROPROSTRANS', 'TORGOVO_RAZVLEK', 
      'FAST_FUD_ZAKUS']),
    ("LEISURE_FREQ", ['ONCE_A_WEEK_AND_LESS', 'TWO_THR_TIMES_M', 'ONCE_A_MONTH', 'NOT_EVERY_MONTH'],
     ['KINOTEATRY', 'TEATRY_KLASSICH', 'BARY_NOCHNYE_KL', 'CONCERTS24', 'IGROVYE_TSENTRY_', 'SALONY_KRASOTY', 'KURSY_MASTER_K', 
      'PARKIPLOHADI', 'SPORTIVNYE_TSENT', 'SPORTIVNYE_SORE', 'KAFE_RESTORANY', 'KOFEJNI', 'GASTROPROSTRANS', 'TORGOVO_RAZVLEK', 
      'FAST_FUD_ZAKUS'])
]

df2 = bp.calc_project_category_universe(project_ids=profile_id, demo=demo, properties=properties)
df2

Считаем размер сэмпла

In [ ]:
# добавляем расчет сэмпла
sample = bp.calc_sample(project_ids=profile_id, demo=demo)
sample

Считаем размер universe по всему профилю и по целевой группе

In [ ]:
# добавляем расчет юниверс и тотал юнивес
total_universe = bp.calc_universe(project_ids=profile_id, demo=None)

universe = bp.calc_universe(project_ids=profile_id, demo=demo)

print(total_universe)
print(universe)

Добавляем к рассчитанному universe расчет долей (col%)

In [ ]:
# приводим первый датафрейм к единому виду со вторым (добавляем недостающие колонки)
df1.insert(loc=df1.columns.get_loc('property_text') + 1, column='category_sys_name', value='')
df1.insert(loc=df1.columns.get_loc('category_sys_name') + 1, column='category_text', value='')

# удаляем колонку сэмпла как ненужную
df1.drop(columns=['sample'], inplace=True)
df2.drop(columns=['sample'], inplace=True)

# добавляем расчет доли
df_tmp = pd.concat([df1, df2], ignore_index=True)
df_tmp['universe_col%'] = round(100 * df_tmp['universe'] / universe, 1)

# добавляем строку с итогами
total_row = pd.DataFrame({
    'property_sys_name': [''],
    'property_text': [''],
    'category_sys_name': [''],
    'category_text': [''],
    'option_sys_name': [''],
    'option_text': ['total'],
    'universe': [universe],
    'universe_col%': [100]
})

result = pd.concat([total_row, df_tmp], 
                   ignore_index=True)
result

Сохраняем в excel с формитированием отчета

In [ ]:
# сохранение в эксель с форматированием
import os

subfolder = "excel"  # название подпапки
os.makedirs(subfolder, exist_ok=True)  # создаёт подпапку, если не существует

filepath = os.path.join(subfolder, "02_crosstab_profile_aud_dosug.xlsx")

with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
    result.to_excel(writer, sheet_name='Расчет', startrow=8, startcol=1, index=False)
    
    # настройка листа экселя
    worksheet = writer.sheets['Расчет']
    
    # Устанавливаем ширину столбцов (учитываем startcol=1)
    worksheet.column_dimensions['B'].width = 20  
    worksheet.column_dimensions['C'].width = 20 
    worksheet.column_dimensions['D'].width = 20  
    worksheet.column_dimensions['E'].width = 20 
    worksheet.column_dimensions['F'].width = 20  
    worksheet.column_dimensions['J'].width = 20  
    
    worksheet['A1'] = 'База данных: Brandpulse 2024'
    worksheet['A2'] = f'Размер генеральной совокупности (тыс.): {total_universe}'
    worksheet['A3'] = 'Целевая база: Население <2024 год> [Профиль]'
    worksheet['A4'] = f'Размер целевой базы (тыс.): {total_universe}'
    worksheet['A5'] = 'Целевая группа: Ж 25-50'
    worksheet['A6'] = f'Размер целевой группы (тыс.): {universe}     Выборка: {sample}'
    worksheet['A7'] = f'Размер (%): {round(100 * universe / total_universe, 1)}%'
    
    worksheet['B9'] = ''
    worksheet['C9'] = ''
    worksheet['D9'] = ''
    worksheet['E9'] = ''
    worksheet['F9'] = ''
    worksheet['G9'] = ''
    worksheet['I9'] = 'col%'